# Sawtooth data audit: is ChatGPT's summary correct?

ChatGPT summarized Kieran's Toast exports plus several datasets built from them (an event register, a joined daily table, a forecasting audit). This notebook re-derives every claim that can be checked against the raw exports in `data/sawtooth/`, and flags the claims that depend on files that aren't in this repo.

**How to review it:** each section loads what it needs, prints the evidence, and records a verdict with `check(...)`. Section 10 collects every verdict into one scorecard.

**How to run it:** from `eda/`, run `uv sync`, then `uv run jupyter lab`. The raw exports must be in `data/sawtooth/` (they're gitignored; see `data/README.md`).

## Verdict

**The numbers ChatGPT pulled from Kieran's exports check out. Its reading of the gaps doesn't, and it missed the biggest pattern in the data.** Section 10 has the full scorecard.

| | |
|---|---|
| **Correct** | File counts; 659 dates, 548 with sales, 111 without; all 22 monthly averages; Event = 10.6% of net sales; no daily item data in Product Mix |
| **Wrong framing** | The 111 "missing" days aren't missing. The daily rows reconcile to Toast's period totals to the cent, so those days had $0 sales: closed Sundays, closed Mondays from late May 2026, and a few holidays. "Hours and closures unresolved" overstates the gap |
| **Omitted** | Net sales for Jan 1 to Oct 21 are down 22.6% year over year, while orders are down only 3.5%. 17-28% of net sales are free-text "open items" with no menu-item ID. Single days that look like private bookings spike to $35.6K against a $3.3K median open day |
| **Can't verify here** | The event register (971 / 797 / 736 + 61), the 659 x 34 forecasting dataset, and the forecasting model, because those files aren't in `data/`. Their arithmetic is internally consistent, and the reported 35.2% WAPE does beat naive baselines (about 43%) on the same 82 test days |

In [ ]:
from pathlib import Path
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Patch, Rectangle

warnings.filterwarnings("ignore", message="Workbook contains no default style")

DATA = Path("../data/sawtooth")
assert DATA.exists(), "Put Kieran's exports in data/sawtooth/ (see data/README.md)"

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

# Colors from the dataviz reference palette (light mode). Blue/orange pair validated for CVD.
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE = "#2a78d6", "#eb6834"
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "figure.dpi": 110,
})

SCORECARD = []


def check(claim, chatgpt, actual, verdict, note=""):
    # Records one claim for the scorecard in section 10.
    SCORECARD.append({"claim": claim, "chatgpt said": chatgpt, "data says": actual, "verdict": verdict, "note": note})
    print(f"[{verdict}] {claim}\n    ChatGPT: {chatgpt}\n    Data:    {actual}" + (f"\n    Note:    {note}" if note else ""))

## 1. File inventory

ChatGPT: *"4 Sales Summary files"*, *"4 Product Mix files for 2025–26, plus one 2022 archive"*, *"5 sales-breakdown CSVs"*.

In [ ]:
sales_summaries = sorted(DATA.glob("SalesSummary_*.xlsx"))
product_mixes = sorted(DATA.glob("ProductMix_*.xlsx"))
breakdowns = sorted(DATA.glob("sales-breakdown-*.csv"))
pm_2022 = [p for p in product_mixes if "_2022-" in p.name]

check("Sales Summary exports", "4", str(len(sales_summaries)), "CORRECT")
check("Product Mix exports", "4 for 2025-26 + 1 archive from 2022",
      f"{len(product_mixes) - len(pm_2022)} for 2025-26 + {len(pm_2022)} from 2022 ({pm_2022[0].stem.split('_', 1)[1]})",
      "CORRECT")

In [ ]:
# The CSV rows end in a trailing comma, so index_col=False is required or pandas silently
# shifts every column left and uses "Sales Category" as the index.
bd_files = pd.DataFrame([{"file": p.name, "split by": pd.read_csv(p, index_col=False, nrows=0).columns[1]}
                         for p in breakdowns])
bd_files

In [ ]:
check("Sales-breakdown CSVs", "5", f"{len(breakdowns)} files covering 4 distinct periods", "CORRECT",
      "jan1-jun30-2025.csv and jan1-jun30-2025-2.csv are the same half-year, split by Dining Option vs Service Type.")

## 2. Daily sales: how many days are really missing?

ChatGPT: *"659 calendar dates: January 1, 2025–October 21, 2026"*, *"548 dates with numeric daily sales"*, *"111 dates with missing or blank sales, which aren't assumed to be zero."*

The only per-day data is the `Sales by day` sheet in each Sales Summary export: one row per business date with Net sales, Total orders, and Total guests.

In [ ]:
def load_daily():
    frames = []
    for f in sales_summaries:
        d = pd.read_excel(f, sheet_name="Sales by day")
        d["source"] = f.name
        frames.append(d)
    d = pd.concat(frames, ignore_index=True)
    d["date"] = pd.to_datetime(d["yyyyMMdd"].astype(str), format="%Y%m%d")
    return d.drop(columns="yyyyMMdd").rename(
        columns={"Net sales": "net_sales", "Total orders": "orders", "Total guests": "guests"})


daily = load_daily()
assert not daily.date.duplicated().any(), "overlapping export periods"

# One row per calendar day. `status` says why a day has no numeric sales.
cal = pd.DataFrame({"date": pd.date_range(daily.date.min(), daily.date.max())}).merge(daily, on="date", how="left")
cal["status"] = np.select([cal.source.isna(), cal.net_sales.isna()],
                          ["no Toast row", "row with blank sales"], "numeric sales")
cal["weekday"] = cal.date.dt.day_name()

print(f"Toast rows: {len(daily)}  |  calendar days: {len(cal)} ({cal.date.min():%Y-%m-%d} to {cal.date.max():%Y-%m-%d})")
cal.status.value_counts()

In [ ]:
n_cal, n_num = len(cal), int(cal.net_sales.notna().sum())
check("Calendar span", "659 dates, Jan 1 2025 - Oct 21 2026",
      f"{n_cal} dates, {cal.date.min():%b %d %Y} - {cal.date.max():%b %d %Y}", "CORRECT")
check("Dates with numeric sales", "548", str(n_num), "CORRECT")
check("Dates without numeric sales", "111", f"{n_cal - n_num} ({(cal.status == 'no Toast row').sum()} with no row + "
      f"{(cal.status == 'row with blank sales').sum()} rows with blank sales)", "CORRECT",
      "The count is right. Whether these are 'missing' is tested next.")

### Reconciliation: do the daily rows add up to Toast's own period totals?

If the daily rows sum to each export's Net sales total to the cent, then the days with no row (and the blank rows) contributed **$0**. They are known zero-sales days, not unknown values.

In [ ]:
rec = []
for f in sales_summaries:
    period_total = pd.read_excel(f, sheet_name="Net sales summary")["Net sales"].iloc[0]
    rows = daily[daily.source == f.name]
    rec.append({"export": f.stem, "daily rows": len(rows), "sum of daily net sales": rows.net_sales.sum(),
                "export's Net sales total": period_total})
rec = pd.DataFrame(rec)
rec["difference"] = (rec["sum of daily net sales"] - rec["export's Net sales total"]).round(2)
rec.style.format({"sum of daily net sales": "${:,.2f}", "export's Net sales total": "${:,.2f}", "difference": "${:,.2f}"})

In [ ]:
check("Treating the 111 days as missing (not zero)", "missing or blank; not assumed to be zero",
      f"daily rows reconcile to period totals exactly (max |diff| ${rec.difference.abs().max():.2f}); the 111 days had $0 net sales",
      "WRONG FRAMING",
      "They are closures and near-empty days, not gaps in the data. Excluding them is fine for an average per open day, "
      "but calling them missing hides the closure schedule (section 3).")

## 3. What the 111 non-numeric days actually are

ChatGPT listed *"hours and closures unresolved"* as a gap. The data resolves most of it.

In [ ]:
nonnum = cal[cal.status != "numeric sales"]
pd.crosstab(nonnum.weekday, nonnum.status, margins=True).reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday", "All"]).fillna(0).astype(int)

In [ ]:
# Every non-Sunday day without numeric sales
nonnum[nonnum.weekday != "Sunday"][["date", "weekday", "status", "orders"]].reset_index(drop=True)

**Mondays changed in mid-2026.** Below: how many Mondays each month had numeric sales. Mondays were regular trading days through spring 2026, then essentially stopped.

In [ ]:
mon = cal[cal.date.dt.day_of_week == 0]
mon.groupby(mon.date.dt.to_period("M")).agg(
    mondays=("date", "size"),
    with_sales=("net_sales", lambda s: int(s.notna().sum())),
    median_sales=("net_sales", "median"),
).style.format({"median_sales": "${:,.0f}"}, na_rep="-")

**Sundays are closed** except for a handful of days with very few orders but large totals, which look like private bookings (look at net sales per order).

In [ ]:
sundays_open = cal[(cal.weekday == "Sunday") & cal.net_sales.notna()].copy()
sundays_open["net per order"] = sundays_open.net_sales / sundays_open.orders
sundays_open[["date", "net_sales", "orders", "guests", "net per order"]].style.format(
    {"date": "{:%Y-%m-%d}", "net_sales": "${:,.0f}", "orders": "{:.0f}", "guests": "{:.0f}", "net per order": "${:,.0f}"})

In [ ]:
n_sun = int((nonnum.weekday == "Sunday").sum())
n_mon_2026 = int(((nonnum.weekday == "Monday") & (nonnum.date >= "2026-05-25")).sum())
n_other = len(nonnum) - n_sun - n_mon_2026
check("Hours and closures", "unresolved",
      f"inferable from the {len(nonnum)} days: {n_sun} Sundays, {n_mon_2026} Mondays from May 25 2026 on, "
      f"{n_other} others (Dec 24-25, Jul 4, one Monday in Jan 2026)",
      "PARTLY WRONG",
      "Still worth confirming with Kieran (closed vs POS outage, and why Mondays stopped), but the pattern is in the data.")

## 4. ChatGPT's monthly chart

ChatGPT plotted the monthly average of observed daily net sales and listed 22 values. Below they're recomputed and compared. A value matches if ChatGPT's figure equals the true mean rounded to $0.1K.

In [ ]:
CHATGPT_MONTHLY_K = [4.1, 4.7, 5.5, 5.0, 5.2, 5.7, 2.9, 5.6, 4.1, 4.7, 5.2, 4.3,   # 2025
                     3.5, 4.3, 4.4, 3.7, 5.0, 4.5, 2.9, 3.5, 4.0, 4.4]              # 2026 (Oct = 1-21)

numeric = cal.dropna(subset=["net_sales"])
monthly = numeric.groupby(numeric.date.dt.to_period("M")).net_sales.agg(open_days="count", mean="mean", total="sum")
monthly["chatgpt ($K)"] = CHATGPT_MONTHLY_K
monthly["recomputed ($K)"] = monthly["mean"] / 1000
monthly["match"] = (monthly["chatgpt ($K)"] - monthly["recomputed ($K)"]).abs() <= 0.05 + 1e-9
monthly.style.format({"mean": "${:,.0f}", "total": "${:,.0f}", "chatgpt ($K)": "{:.1f}", "recomputed ($K)": "{:.3f}"})

In [ ]:
check("22 monthly averages of observed daily sales", "values listed in the chart",
      f"{int(monthly.match.sum())} of {len(monthly)} match to rounding", "CORRECT",
      "The pasted chart's lowest y tick was $2.6K (not zero-based), which visually exaggerates swings. The numbers are right.")

### What the per-open-day average hides

An average per open day is a fair "how busy is a typical open night" number. It hides two things: fewer open days in 2026 (closed Mondays), and a year-over-year decline. Monthly **totals**, same calendar window in both years (October cut to the 1st through 21st), show both.

In [ ]:
win = cal[(cal.date.dt.month <= 10) & ~((cal.date.dt.month == 10) & (cal.date.dt.day > 21))]
tot = win.groupby([win.date.dt.month, win.date.dt.year]).net_sales.sum().unstack() / 1000
labels = [pd.Timestamp(2025, m, 1).strftime("%b") for m in tot.index]
labels[-1] = "Oct 1-21"

fig, ax = plt.subplots(figsize=(10, 4))
x, w = np.arange(len(tot)), 0.38
ax.bar(x - w / 2 - 0.01, tot[2025], w, color=BLUE, label="2025")
ax.bar(x + w / 2 + 0.01, tot[2026], w, color=ORANGE, label="2026")
ax.set_xticks(x, labels)
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"${v:,.0f}K"))
ax.set_title("Monthly net sales, 2025 vs 2026 (same calendar window)")
ax.grid(axis="x", visible=False)
ax.legend(loc="upper right", ncols=2)
plt.show()

ytd = win.groupby(win.date.dt.year).agg(net_sales=("net_sales", "sum"), open_days=("net_sales", "count"),
                                        orders=("orders", "sum"), guests=("guests", "sum")).astype(float)
ytd["net per order"] = ytd.net_sales / ytd.orders
ytd.loc["change"] = ytd.loc[2026] / ytd.loc[2025] - 1
ytd.style.format(lambda v: f"{v:+.1%}" if abs(v) < 1 else f"{v:,.0f}")

In [ ]:
yoy = ytd.loc["change", "net_sales"]
check("Overall sales trend", "not mentioned",
      f"Jan 1-Oct 21 net sales {yoy:+.1%} year over year; open days {ytd.loc['change', 'open_days']:+.1%}, "
      f"orders {ytd.loc['change', 'orders']:+.1%}, net per order {ytd.loc['change', 'net per order']:+.1%}",
      "OMITTED",
      "A level shift between years matters for any model trained on 2025 and tested on 2026.")

### Year over year by hour of day

Each Sales Summary export also has a `Time of day (totals)` sheet: net sales by hour of day, totaled over the whole export, with no dates. Its rows add up to each export's net sales total (asserted below). Because it's a period total:

- **Only Jan-Jun is an exact year-over-year match.** In the second half, 2025 covers Jul-Dec and 2026 stops at Oct 21, and these totals can't be cut to the same dates. Read that panel for shape, not level.
- **Bars are each hour's share of the period's net sales**, so periods of different lengths compare. Hours run in business-day order, since Toast's Late Night daypart runs to 2 AM. The handful of sales from 3 to 9 AM share one bar.

In [ ]:
def hour_slot(h):
    # 3-9 AM folds into one pre-opening bar; 12-2 AM stays at the end of the night (Toast's Late Night runs to 2 AM).
    return "3-9a" if 3 <= h < 10 else f"{h % 12 or 12}{'a' if h < 12 else 'p'}"


SLOTS = ["3-9a"] + [hour_slot(h) for h in [*range(10, 24), 0, 1, 2]]

frames = []
for f in sales_summaries:
    h = pd.read_excel(f, sheet_name="Time of day (totals)")
    period_total = pd.read_excel(f, sheet_name="Net sales summary")["Net sales"].iloc[0]
    assert abs(h["Net sales"].sum() - period_total) < 0.005, f"{f.name}: hourly rows don't sum to the period total"
    start = pd.Timestamp(f.stem.split("_")[1])
    frames.append(h.assign(export=f.stem.replace("SalesSummary_", ""), year=start.year,
                           half="Jan-Jun" if start.month == 1 else "Jul-Dec"))
hourly = pd.concat(frames, ignore_index=True).rename(
    columns={"Hour of day": "hour", "Net sales": "net_sales", "Total orders": "orders"})
hourly["net_sales"] = hourly.net_sales.fillna(0)  # blank = $0, since the rows reconcile to the totals (assert above)
hourly["slot"] = pd.Categorical(hourly.hour.map(hour_slot), SLOTS, ordered=True)

by_slot = hourly.groupby(["half", "year", "slot"], observed=False).net_sales.sum()
hour_share = by_slot / by_slot.groupby(["half", "year"]).transform("sum")

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, sharey=True, constrained_layout=True)
titles = {"Jan-Jun": "Jan-Jun, 2025 vs 2026 (same window)",
          "Jul-Dec": "Jul-Dec 2025 vs Jul 1-Oct 21 2026 (windows differ: compare shape, not level)"}
x, w = np.arange(len(SLOTS)), 0.38
for ax, (half, title) in zip(axes, titles.items()):
    ax.bar(x - w / 2 - 0.01, hour_share[half, 2025], w, color=BLUE, label="2025")
    ax.bar(x + w / 2 + 0.01, hour_share[half, 2026], w, color=ORANGE, label="2026")
    ax.set_title(title)
    ax.grid(axis="x", visible=False)
axes[-1].set_xticks(x, SLOTS)
axes[0].yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1, decimals=0))
axes[0].legend(loc="upper left", ncols=2)
fig.supylabel("Share of the period's net sales", color=INK2, fontsize=10)
plt.show()

In [ ]:
h1_change = by_slot["Jan-Jun", 2026].sum() / by_slot["Jan-Jun", 2025].sum() - 1
print(f"Jan-Jun net sales, 2026 vs 2025: {h1_change:+.1%} (read each hour's $ change against this)")

tbl = hour_share.unstack(["half", "year"])[[("Jan-Jun", 2025), ("Jan-Jun", 2026), ("Jul-Dec", 2025), ("Jul-Dec", 2026)]]
tbl.columns = ["Jan-Jun 2025", "Jan-Jun 2026", "Jul-Dec 2025", "Jul 1-Oct 21 2026"]
tbl.insert(2, "Jan-Jun $ change", by_slot["Jan-Jun", 2026] / by_slot["Jan-Jun", 2025] - 1)
display(tbl.style.format("{:.1%}", na_rep="-").format("{:+.0%}", subset="Jan-Jun $ change", na_rep="-")
        .set_caption("Share of each period's net sales by hour, plus the dollar change for the matched Jan-Jun window"))

# Hours where the average order is $150+ (the same cut section 11 uses for big-ticket days)
big = hourly.assign(per_order=hourly.net_sales / hourly.orders).query("per_order >= 150")
display(big[["export", "hour", "net_sales", "orders", "per_order"]].sort_values(["export", "hour"]).style.format(
    {"net_sales": "${:,.0f}", "per_order": "${:,.0f}"}).hide(axis="index").set_caption("Hours averaging $150+ per order"))

**What the hourly view shows**

- **Jan-Jun: the shape of a night barely moved, and the decline is concentrated late.** 5-7 PM is 42-44% of net sales in both years. Against a 20.1% drop overall, 11 PM fell 45% and midnight 51%. From 5 PM on, 9 PM (+10%) is the only hour that grew.
- **Jul 1-Oct 21, 2026: lunch is gone.** 11 AM-2 PM falls from 8.5% of net sales (Jul-Dec 2025) to 0.8%, and Toast's `Service Daypart summary` sheet agrees: 46 Lunch orders, against 1,303 in Jan-Jun 2026 and 1,502 in Jul-Dec 2025. The mismatched windows can't explain a drop that size. Late night (9 PM-2 AM) rises from 25% to 40% of net sales. Lunch still ran in at least part of Jan-Jun 2026, so it stopped sometime in 2026, like Mondays did; these exports can't say exactly when. Worth confirming with Kieran. It also means the forecast test window in section 8 (Jul 1-Oct 21, 2026) had no lunch service, while the training data did.
- **1-2 AM and a few daytime hours are dominated by large tickets.** They average $196 to $2,722 per order, against a median of $42 per order across open days (section 6). That fits deposits or bookings rung in outside service (the Event category also takes deposits for future events; see `data/README.md`). They're real net sales, so they stay in the shares.

## 5. Daily view (calendar heatmap)

One cell per day, Monday at the top. Darker is higher net sales. The color scale caps at $10K so normal nights stay readable; the few days above that are listed in section 6. Hatched cells had no numeric sales.

In [ ]:
VMAX = 10_000
cmap = LinearSegmentedColormap.from_list("blue_seq", BLUE_RAMP)
norm = Normalize(0, VMAX)
HATCH = {"no Toast row": "////", "row with blank sales": "xxxx"}

fig, axes = plt.subplots(2, 1, figsize=(13, 4.6), constrained_layout=True)
for ax, year in zip(axes, [2025, 2026]):
    start = pd.Timestamp(year, 1, 1)
    offset = start.day_of_week
    for r in cal[cal.date.dt.year == year].itertuples():
        col, row = ((r.date - start).days + offset) // 7, r.date.day_of_week
        if r.status == "numeric sales":
            ax.add_patch(Rectangle((col, row), 0.9, 0.9, facecolor=cmap(norm(r.net_sales)), linewidth=0))
        else:
            ax.add_patch(Rectangle((col, row), 0.9, 0.9, facecolor=SURFACE, edgecolor="none",
                                   hatch=HATCH[r.status], hatchcolor=MUTED, linewidth=0))
    months = pd.date_range(start, f"{year}-12-01", freq="MS")
    ax.set_xticks([((m - start).days + offset) // 7 + 0.45 for m in months], [m.strftime("%b") for m in months])
    ax.set_yticks(np.arange(7) + 0.45, ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
    ax.set_xlim(0, 53.5); ax.set_ylim(7, 0); ax.set_aspect("equal")
    ax.grid(False); ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(f"{year}" + (" (data ends Oct 21)" if year == 2026 else ""))

fig.colorbar(ScalarMappable(norm, cmap), ax=axes, extend="max", shrink=0.7, label="Net sales per day",
             format=mpl.ticker.FuncFormatter(lambda v, _: f"${v / 1000:.0f}K"))
fig.legend(handles=[Patch(facecolor=SURFACE, hatch=h, edgecolor=MUTED, label=s) for s, h in HATCH.items()],
           loc="lower left", ncols=2, bbox_to_anchor=(0.0, -0.08))
plt.show()

## 6. Toast's "Event" category and lumpy private bookings

ChatGPT: *"That category represents 10.6% of reported revenue."* The sales-breakdown CSVs have period totals by category; the category-total rows are the ones with an empty Service Type.

In [ ]:
PERIOD = {
    "sales-breakdown-jan1-jun30-2025-2.csv": "2025 H1",
    "sales-breakdown-jul1-dec31-2025.csv": "2025 H2",
    "sales-breakdown-jan1-jun30-2026.csv": "2026 H1",
    "sales-breakdown-jul1-oct21-2026.csv": "2026 Jul 1-Oct 21",
}  # the Dining Option split of 2025 H1 is left out so that period isn't counted twice

cat_totals = {}
for name, label in PERIOD.items():
    df = pd.read_csv(DATA / name, index_col=False)
    cat_totals[label] = df[df["Service Type"].isna()].set_index("Sales Category")["Net Sales"]
cat_totals = pd.DataFrame(cat_totals)

ev = pd.DataFrame({"Event net sales": cat_totals.loc["Event"], "all categories": cat_totals.sum()})
ev.loc["all periods"] = ev.sum()
ev["Event share"] = ev["Event net sales"] / ev["all categories"]
ev.style.format({"Event net sales": "${:,.0f}", "all categories": "${:,.0f}", "Event share": "{:.1%}"})

In [ ]:
share = ev.loc["all periods", "Event share"]
check("Event category share of revenue", "10.6%", f"{share:.1%} of net sales across all four periods", "CORRECT",
      "Share varies a lot by period (see table), and these rows have no dates, so the daily split ChatGPT asked Kieran for is genuinely needed.")

Event revenue is lumpy: a few bookings land on single days and look like demand spikes. These are the biggest days and the days with the highest net sales per order.

In [ ]:
days = numeric.assign(per_order=numeric.net_sales / numeric.orders)
fmt = {"date": "{:%Y-%m-%d}", "net_sales": "${:,.0f}", "orders": "{:.0f}", "guests": "{:.0f}", "per_order": "${:,.0f}"}
cols = ["date", "weekday", "net_sales", "orders", "guests", "per_order"]
print(f"Median open day: ${days.net_sales.median():,.0f}, ${days.per_order.median():,.0f} per order")
display(days.nlargest(10, "net_sales")[cols].style.format(fmt).set_caption("Top 10 days by net sales"))
display(days.nlargest(8, "per_order")[cols].style.format(fmt).set_caption("Top 8 days by net sales per order"))

## 7. Product Mix: is there daily item-level data?

ChatGPT: *"Product Mix includes sales and quantities by item, totaled over each export period. It doesn't provide daily sales per item."*

In [ ]:
pm_rows = []
for p in product_mixes:
    sheets = pd.read_excel(p, sheet_name=None)
    date_like = [f"{s}: {c}" for s, df in sheets.items() for c in df.columns
                 if "date" in str(c).lower() or pd.api.types.is_datetime64_any_dtype(df[c])]
    pm_rows.append({"export": p.stem, "sheets": len(sheets), "rows in Items": len(sheets["Items"]),
                    "date-like columns": ", ".join(date_like) or "none"})
pd.DataFrame(pm_rows)

In [ ]:
check("Product Mix granularity", "item qty and sales per export period; no daily item sales",
      "no date column in any sheet of any Product Mix export", "CORRECT")

**Not mentioned by ChatGPT: open items.** Toast "open items" are free-text entries typed by staff, with no menu-item ID. Only the `Items` sheet carries a stable item identity, so the open-item share caps what item-level modeling can learn.

In [ ]:
oi_rows = []
for p in product_mixes:
    if p in pm_2022:
        continue
    items = pd.read_excel(p, sheet_name="Items")
    open_items = pd.read_excel(p, sheet_name="Open items")
    item_net = items.loc[items["Sales Category"].notna(), "Net item amt"].sum()
    open_net = open_items.loc[open_items["Sales Category"].notna(), "Net item amt"].sum()
    open_ex_event = open_items.loc[open_items["Sales Category"].notna() & (open_items["Sales Category"] != "Event"),
                                   "Net item amt"].sum()
    oi_rows.append({"export": p.stem.replace("ProductMix_", ""), "menu items net": item_net, "open items net": open_net,
                    "open share": open_net / (item_net + open_net),
                    "open share excl. Event": open_ex_event / (item_net + open_ex_event)})
pd.DataFrame(oi_rows).style.format({"menu items net": "${:,.0f}", "open items net": "${:,.0f}",
                                    "open share": "{:.1%}", "open share excl. Event": "{:.1%}"})

Ticketed shows do carry dates, but only inside the item **name** (for example "7/8 8PM VALLEY IMPROV"). That's the source of ChatGPT's *"ticket-label event leads."* The names have no year, and the date is the **show** date, not the sale date. The Jan-Jun 2026 export already sold tickets for a 7/8 show, most likely a presale. That's why ChatGPT was right to flag these for confirmation.

In [ ]:
tick_rows = []
for p in product_mixes:
    al = pd.read_excel(p, sheet_name="All levels")
    t = al[(al["Type"] == "menuItem") & (al["Menu"] == "TICKETS") & (al["Qty sold"] > 0)]
    dated = t["Item, open item"].astype(str).str.contains(r"\b\d{1,2}/\d{1,2}\b")
    tick_rows.append({"export": p.stem.replace("ProductMix_", ""), "ticket items sold": len(t),
                      "with a date in the name": int(dated.sum()),
                      "examples": " | ".join(t.loc[dated, "Item, open item"].head(3))})
pd.DataFrame(tick_rows)

## 8. ChatGPT's forecasting audit

ChatGPT: *"an initial seasonal model with approximately $1,275 daily MAE and 35.2% WAPE over 82 observed test dates."* The audit file isn't in the repo, so the model can't be re-run. Three things can still be checked:

1. Does an 82-date test set fit the data?
2. Are MAE and WAPE consistent with each other? (WAPE = total absolute error / total actual sales = MAE x n / sum of actuals.)
3. How does that compare with trivial baselines on the same dates?

In [ ]:
TEST_START, TEST_END = pd.Timestamp("2026-07-01"), pd.Timestamp("2026-10-21")
series = cal.set_index("date").net_sales          # NaN on days without numeric sales
test = series[TEST_START:TEST_END].dropna()
implied_wape = 1275 * len(test) / test.sum()

check("Forecast test set", "82 observed test dates", f"{len(test)} days with numeric sales in Jul 1-Oct 21 2026 (the last export)",
      "CONSISTENT", "Consistent with testing on the last export; can't confirm without the audit file.")
check("MAE vs WAPE", "$1,275 MAE, 35.2% WAPE", f"$1,275 x {len(test)} / ${test.sum():,.0f} = {implied_wape:.1%}",
      "CONSISTENT")

In [ ]:
def score(pred):
    pred = pred.reindex(test.index)
    ok = pred.notna()
    err = (test[ok] - pred[ok]).abs()
    return pd.Series({"days scored": int(ok.sum()), "MAE": err.mean(), "WAPE": err.sum() / test[ok].sum()})


train = series[:TEST_START - pd.Timedelta(days=1)].dropna()
weekday_median = train.groupby(train.index.day_of_week).median()

baselines = {
    "Weekday median of Jan 2025-Jun 2026 (works at any horizon)": pd.Series(test.index.day_of_week.map(weekday_median), index=test.index),
    "Same weekday last week (a 7-day-ahead forecast)": series.shift(7, freq="D"),
    "Same weekday 4 weeks earlier (a 28-day-ahead forecast)": series.shift(28, freq="D"),
}
results = pd.DataFrame({k: score(v) for k, v in baselines.items()}).T
results.loc["ChatGPT's 'initial seasonal model' (as reported)"] = [len(test), 1275, 0.352]
results.style.format({"days scored": "{:.0f}", "MAE": "${:,.0f}", "WAPE": "{:.1%}"})

In [ ]:
naive = results.drop(index="ChatGPT's 'initial seasonal model' (as reported)")
check("Model quality", "initial seasonal model, 35.2% WAPE; event features didn't help",
      f"naive baselines on the same dates: {naive.WAPE.min():.1%} to {naive.WAPE.max():.1%} WAPE",
      "UNVERIFIABLE",
      f"If the reported 35.2% is right it beats the best naive baseline by {(naive.WAPE.min() - 0.352) * 100:.1f} percentage points. "
      "ChatGPT's own caveat holds: it was one-day-ahead, and naive error grows from the 7-day to the 28-day horizon.")

## 9. Event-register claims (source files not in this repo)

The master event register, the date-expanded events, the joined workbook, and the "strict forecasting dataset" aren't in `data/`. Only internal arithmetic can be checked: do ChatGPT's monthly bars add up to its stated totals?

In [ ]:
chatgpt_events = pd.DataFrame({
    "month": pd.period_range("2025-01", "2026-10", freq="M"),
    "eligible": [36, 42, 34, 39, 21, 30, 6, 13, 42, 50, 43, 21, 49, 38, 24, 49, 33, 26, 13, 8, 65, 54],
    "requires review": [0, 0, 2, 4, 3, 3, 2, 6, 2, 2, 1, 1, 1, 0, 0, 3, 2, 3, 2, 17, 5, 2],
})
e, r = chatgpt_events[["eligible", "requires review"]].sum()
check("Event-date records", "736 eligible + 61 review = 797", f"monthly bars sum to {e} + {r} = {e + r}",
      "CONSISTENT", "Arithmetic only. The register itself isn't here to check.")
check("Master event register", "971 records", "file not in repo", "UNVERIFIABLE")
check("Strict forecasting dataset", "659 rows x 34 columns", "file not in repo (659 does match the calendar span)", "UNVERIFIABLE")

## 10. Scorecard

In [ ]:
scorecard = pd.DataFrame(SCORECARD)
scorecard.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}).hide(axis="index")

In [ ]:
scorecard.verdict.value_counts()

## 11. Open decision: how closed days enter the forecasting target

Section 2 showed the 111 non-numeric days are $0 days in Toast's own totals. How they go into the model is a modeling choice, not a data question.

Implemented with options (b) and (c): forecast open days only, label why every other day is closed so closures can come from a schedule (to confirm with Kieran), and flag big-ticket days rather than deleting them. Days under $300 count as closed, since they're effectively closures (Thanksgiving $10, New Year's Day $60). `modeling/demand_model_v0.ipynb` currently keeps those near-empty days in its training data; switching it to this encoding would be a small change.

In [ ]:
NEAR_EMPTY = 300   # $; below this a "trading" day is effectively a closure


def encode_closed_days(cal: pd.DataFrame) -> pd.DataFrame:
    # Options (b) + (c): the target is net sales on open days only (NaN otherwise), every closed day gets a
    # reason so closures can be supplied as a schedule, and big-ticket days are flagged, not removed.
    out = cal[["date", "weekday", "status", "net_sales", "orders", "guests"]].copy()
    out["is_open"] = (out.status == "numeric sales") & (out.net_sales >= NEAR_EMPTY)
    out["closed_reason"] = np.select(
        [out.is_open, out.weekday == "Sunday", (out.weekday == "Monday") & (out.date >= "2026-05-25"),
         out.status == "numeric sales"],
        ["", "Sunday", "Monday (closed from May 25, 2026)", "near-empty day"], "other closure or holiday")
    out["target"] = out.net_sales.where(out.is_open)
    out["big_ticket_day"] = out.is_open & (out.net_sales / out.orders >= 150)
    return out

In [ ]:
encoded = encode_closed_days(cal)
print(f"{int(encoded.is_open.sum())} open days, {int((~encoded.is_open).sum())} closed, "
      f"{int(encoded.big_ticket_day.sum())} big-ticket days flagged among the open ones")
display(encoded.closed_reason.replace("", "open").value_counts().rename("days").to_frame())
encoded[encoded.closed_reason.isin(["near-empty day", "other closure or holiday"])][
    ["date", "weekday", "status", "net_sales", "orders", "closed_reason"]].style.format(
    {"date": "{:%Y-%m-%d}", "net_sales": "${:,.0f}", "orders": "{:.0f}"}, na_rep="-").hide(axis="index")